# M3 submission preparation: copy the selected validated package
Before execution, select **Accelerator: None**, **Internet: Off**. Kaggle auto-extracts uploaded `.zip` files in the observed dataset workflow. Upload a byte-identical copy of the M2 package named `submission.zip.bin`, together with the unchanged committed `manifest.json`, as a private input. Do not rename or edit the original M2 evidence. Set MANIFEST_PATH and PACKAGE_PATH to the actual mounted files and set AUTHORIZATION below. The input filename does not replace any identity check.

This notebook performs no installation, compilation, extraction, model serving, evaluation or submission API call. The only output write is `/kaggle/working/submission.zip`, copied byte-for-byte. The approval phrase authorizes this preparation; actual competition submission is a separate operator action. Existing output is never overwritten. Stored official sources do not fully document the live UI selection sequence; review it before any submission. A scored submission would execute the agent on competition infrastructure.

Select CANDIDATE explicitly. B2 is disabled until its pins are populated from reconciled evidence and the first M2 result is known without unexplained errors. These operator assertions are not a leaderboard query or remote attestation.


In [ ]:
import hashlib
import json
from pathlib import Path

AUTHORIZATION_REQUIRED = "AUTHORIZE FORGEAI GEMMA4 M3 FIRST SUBMISSION OF M2 PACKAGE ca6a05b4"
MANIFEST_SHA256 = "91660911e8122cf4e6080add5d52972072f69eb13b8e7f048d1dfac108afac37"
ZIP_SHA256 = "78051300426d369e83a788f57bc6284c56c85d07cc32d6de87fbdc49c9da7e2d"
RECEIPT_SHA256 = "ca6a05b46844c86369c7aaef3d989b542c5f3c5f63fcc91c7fce5997742893d9"
ZIP_SIZE = 1566


CANDIDATE_PINS = {
    "m2": {"manifest_sha256": MANIFEST_SHA256, "zip_sha256": ZIP_SHA256,
           "receipt_sha256": RECEIPT_SHA256, "zip_size": ZIP_SIZE,
           "authorization": AUTHORIZATION_REQUIRED},
    # Fill only after B2 reconciliation/pre-commit and understood M2 scoring result.
    "b2": {"manifest_sha256": "102aa8f225aa264fe6cfa66d01b25c3ccb3db88e1541bf8f86da3cfb3ac6bb09",
           "zip_sha256": "fe47a6f7a807d0baef9bc2b55455616b26d56423229e2e82d402d82663f8dadb",
           "receipt_sha256": "d7195f13fc0f4da056e003317d72c6d46f5b5acb0521742515721684d01f383f",
           "zip_size": 1567,
           "authorization": "AUTHORIZE FORGEAI GEMMA4 SUBMISSION OF B2 PACKAGE d7195f13",
           "reconciled": True, "m2_result_understood": True},
    # P1 requires its own reconciled compiler/package evidence; never guess pins.
    "p1": {"manifest_sha256": "b43c9b1e62c6beb8a028cc6963caa6b1196e8e81249e266375f6ab2398b8318f",
           "zip_sha256": "fab297b8bc6cc934a239431bc32679a0f53bfbd5caf711eef13b6079ecff43d9",
           "receipt_sha256": "20aa090e37a2333851a94501cb09db9fc88f9c46ca92c0880f2a10239a64da57",
           "zip_size": 4736,
           "authorization": "AUTHORIZE FORGEAI GEMMA4 SUBMISSION OF P1 PACKAGE 20aa090e",
           "reconciled": True, "m2_result_understood": True},
}


def selected_pins(candidate):
    if candidate not in CANDIDATE_PINS:
        raise ValueError("Unknown explicit candidate")
    pins = CANDIDATE_PINS[candidate]
    for key in ("manifest_sha256", "zip_sha256", "receipt_sha256"):
        value = pins.get(key)
        if not isinstance(value, str) or len(value) != 64 or any(c not in "0123456789abcdef" for c in value):
            raise ValueError("Candidate hash placeholders must be filled from qualified evidence")
    if type(pins.get("zip_size")) is not int or pins["zip_size"] <= 0:
        raise ValueError("Candidate size placeholder")
    if candidate in ("b2", "p1"):
        expected = "AUTHORIZE FORGEAI GEMMA4 SUBMISSION OF " + candidate.upper() + " PACKAGE " + pins["receipt_sha256"][:8]
        if (pins.get("authorization") != expected or pins.get("reconciled") is not True
                or pins.get("m2_result_understood") is not True):
            raise ValueError(candidate.upper() + " reconciliation and understood M2 result required")
        if any(pins[key] == CANDIDATE_PINS["m2"][key]
               for key in ("manifest_sha256", "zip_sha256", "receipt_sha256")):
            raise ValueError(candidate.upper() + " cannot reuse M2 identity pins")
    if candidate == "p1" and any(
        pins[key] == CANDIDATE_PINS["b2"][key]
        for key in ("manifest_sha256", "zip_sha256", "receipt_sha256")
    ):
        raise ValueError("P1 cannot reuse B2 identity pins")
    return pins


def prepare_submission(manifest_path, package_path, output, authorization, candidate="m2"):
    pins = selected_pins(candidate)
    if authorization != pins["authorization"]:
        raise ValueError("Exact maintainer authorization required")
    if manifest_path.is_symlink() or package_path.is_symlink():
        raise ValueError("Evidence files must not be symlinks")
    raw_manifest = manifest_path.read_bytes()
    if hashlib.sha256(raw_manifest).hexdigest() != pins["manifest_sha256"]:
        raise ValueError("Pinned candidate manifest SHA256 mismatch")
    manifest = json.loads(raw_manifest)
    if (manifest.get("state") != "PORTABLE_COMPILE_AND_PACKAGE_PASS"
            or manifest.get("local_receipt_sha256") != pins["receipt_sha256"]
            or manifest.get("zip_sha256") != pins["zip_sha256"]
            or manifest.get("zip_size") != pins["zip_size"]):
        raise ValueError("Candidate package identity mismatch")
    payload = package_path.read_bytes()
    if len(payload) != pins["zip_size"] or hashlib.sha256(payload).hexdigest() != pins["zip_sha256"]:
        raise ValueError("Candidate submission.zip SHA256/size mismatch")
    # Exclusive creation: preserve prior output; no repacking or extraction.
    with output.open("xb") as stream:
        stream.write(payload)
    if hashlib.sha256(output.read_bytes()).hexdigest() != pins["zip_sha256"]:
        raise ValueError("Copied output SHA256 mismatch; do not submit")
    report = {"state": candidate.upper() + "_PACKAGE_COPIED_UNCHANGED", "output": str(output),
              "zip_sha256": pins["zip_sha256"], "zip_size": pins["zip_size"],
              "manifest_sha256": pins["manifest_sha256"], "receipt_sha256": pins["receipt_sha256"]}
    print(json.dumps(report, indent=2, sort_keys=True))
    return report


In [ ]:
CANDIDATE = "m2"  # Explicit selection: "m2" or "b2"; B2 pins remain blocked.
AUTHORIZATION = ""
MANIFEST_PATH = Path("/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_M2_INPUT/manifest.json")
PACKAGE_PATH = Path("/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_M2_INPUT/submission.zip.bin")
OUTPUT = Path("/kaggle/working/submission.zip")
report = prepare_submission(MANIFEST_PATH, PACKAGE_PATH, OUTPUT, AUTHORIZATION, candidate=CANDIDATE)


Expected: `M2_PACKAGE_COPIED_UNCHANGED` and ZIP SHA256 `78051300426d369e83a788f57bc6284c56c85d07cc32d6de87fbdc49c9da7e2d`. Save the notebook output and hash. Do not execute the official starter cells. After separate maintainer approval, use the competition interface to select this saved notebook output archive if supported; stop if the interface requirements differ. Do not select compiler logs or submission.parquet. Preserve the submitted notebook version, submission ID, timestamps, public score/status and all available scoring logs.
